# Week 8.2 — Agentic deal pipeline (course framework)

This notebook uses **`DealAgentFramework`**: Chroma-backed products, RSS deals via the planning agent, ensemble pricing, and `memory.json`. It follows the **week 8 capstone path** from the course exercise template (same repo as `week8/deal_agent_framework.py`).

A separate notebook in this repo, **`week8_Exercise.ipynb`**, shows RAG + blend pricers on `Item` test data; this one focuses on the live deal pipeline instead.

**If you run a full pipeline:** `OPENAI_API_KEY`. Chroma `products_vectorstore` under `week8/`. Modal logged in with **`pricer-service`** deployed (`SpecialistAgent`). Network for Dealnews RSS. Optional: `HF_TOKEN` if your env needs it for embeddings.

## Setup

Find `week8/`, add it to `sys.path`, and `chdir` there so `memory.json` and Chroma paths match the course layout (works from repo root, `week8/`, or this folder).

In [ ]:
import os
import sys
from pathlib import Path
from typing import Optional

def _find_week8_root(start: Path) -> Optional[Path]:
    for p in [start, *start.parents]:
        if (p / "deal_agent_framework.py").is_file():
            return p
        if (p / "week8" / "deal_agent_framework.py").is_file():
            return p / "week8"
    return None

WEEK8_ROOT = _find_week8_root(Path.cwd().resolve())
if WEEK8_ROOT is None:
    raise FileNotFoundError("Cannot find deal_agent_framework.py")
if str(WEEK8_ROOT) not in sys.path:
    sys.path.insert(0, str(WEEK8_ROOT))
os.chdir(WEEK8_ROOT)

from dotenv import load_dotenv
from deal_agent_framework import DealAgentFramework

load_dotenv(override=True)
print("week8 root:", WEEK8_ROOT)

## Load framework and inspect memory

Opportunities are persisted in **`memory.json`** in the current working directory (`week8/` after the setup cell).

In [ ]:
framework = DealAgentFramework()
print("Opportunities in memory:", len(framework.memory))
if framework.memory:
    o = framework.memory[0]
    desc = o.deal.product_description
    preview = desc if len(desc) <= 220 else desc[:220] + "..."
    print("Sample:", preview)
    print(f"Listed ${o.deal.price:.2f} | estimate ${o.estimate:.2f} | discount ${o.discount:.2f}")

## Optional: one planning cycle

`DealAgentFramework.run()` calls `PlanningAgent.plan`, which scans RSS, prices deals with the ensemble, and **appends to memory** only when the best discount exceeds **`PlanningAgent.DEAL_THRESHOLD`** (default **$50**).

Set `RUN_PIPELINE = True` when your environment is ready.

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    if "framework" not in globals():
        framework = DealAgentFramework()
    n_before = len(framework.memory)
    memory = framework.run()
    print("Stored opportunities:", len(memory))
    if len(memory) > n_before:
        print("New opportunity discount:", f"${memory[-1].discount:.2f}")
    else:
        print("No new opportunity above threshold this run.")
else:
    print("Skipped (set RUN_PIPELINE = True to run the agent loop).")

## Gradio UI: "The Price is Right"

**Terminal** (kernel stays free):

```bash
python price_is_right.py
```

Run from **`week8/`** (this notebook already `chdir`s there in the setup cell).

**In this notebook:** use the next cell. Set **`LAUNCH_UI = True`** and run it to call `App().run()`. That **blocks the kernel** until you stop it (Kernel → Interrupt). Leave **`LAUNCH_UI = False`** to skip.

In [ ]:
LAUNCH_UI = False  # True: start Gradio here (blocks kernel until interrupt)

if LAUNCH_UI:
    import importlib

    # Reload week8 modules so edits to deal_agent_framework.py apply without restarting kernel
    import deal_agent_framework
    import price_is_right

    importlib.reload(deal_agent_framework)
    importlib.reload(price_is_right)

    from price_is_right import App

    App().run()
else:
    print("Gradio not started. Set LAUNCH_UI = True above, or run: python price_is_right.py in a terminal.")